# AI-Powered Invoice Extraction System
### Powered by FireRedTeam/FireRed-OCR + Gradio

Run each cell from top to bottom in order.
The last cell will give you a public Gradio link that works in Colab.

---

## Cell 1 - Install Dependencies

Run this once. Installs all required packages and clones the FireRed-OCR repository.
You do not need to restart the kernel after this step.

In [2]:
import subprocess
import sys
import os

packages = [
    "gradio",
    "transformers",
    "torch",
    "qwen-vl-utils",
    "Pillow",
    "pdf2image",
    "plotly",
    "pandas",
    "accelerate",
    "einops",
]

for pkg in packages:
    print(f"Installing {pkg} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All Python packages installed.")

# Install poppler which is required by pdf2image to convert PDFs to images
print("Installing poppler-utils ...")
subprocess.check_call(["apt-get", "install", "-y", "-q", "poppler-utils"])
print("poppler-utils installed.")

# Clone the FireRed-OCR repository if it is not already present
if not os.path.exists("FireRed-OCR"):
    print("Cloning FireRed-OCR repository ...")
    subprocess.check_call(["git", "clone", "https://github.com/FireRedTeam/FireRed-OCR.git"])
    print("FireRed-OCR cloned successfully.")
else:
    print("FireRed-OCR repository already exists, skipping clone.")

print("\nCell 1 complete. Proceed to Cell 2.")

Installing gradio ...
Installing transformers ...
Installing torch ...
Installing qwen-vl-utils ...
Installing Pillow ...
Installing pdf2image ...
Installing plotly ...
Installing pandas ...
Installing accelerate ...
Installing einops ...
All Python packages installed.
Installing poppler-utils ...
poppler-utils installed.
Cloning FireRed-OCR repository ...
FireRed-OCR cloned successfully.

Cell 1 complete. Proceed to Cell 2.


## Cell 2 - Imports and Model Loading

Loads the FireRed-OCR vision-language model from HuggingFace.
The first run downloads model weights which are around 5 GB.
Subsequent runs load from the local cache and are much faster.

In [3]:
import os
import sys
import json
import uuid
import re
import tempfile
from datetime import datetime
from pathlib import Path

import torch
import pandas as pd
import plotly.express as px
import gradio as gr
from PIL import Image
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

# Add the cloned FireRed-OCR folder to the Python path
# so we can import conv_for_infer from it
FIRERED_PATH = os.path.abspath("FireRed-OCR")
if FIRERED_PATH not in sys.path:
    sys.path.insert(0, FIRERED_PATH)

try:
    from conv_for_infer import generate_conv
    print("conv_for_infer imported successfully.")
except ImportError as e:
    print(f"Warning: could not import conv_for_infer - {e}")
    print("Make sure FireRed-OCR was cloned correctly in Cell 1.")
    generate_conv = None

MODEL_ID = "FireRedTeam/FireRed-OCR"
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE    = torch.bfloat16 if torch.cuda.is_available() else torch.float32

print(f"Running on device: {DEVICE}")
print(f"Loading model: {MODEL_ID}")
print("This may take several minutes on the first run while model weights are downloaded.")

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("FireRed-OCR model loaded successfully.")
print("\nCell 2 complete. Proceed to Cell 3.")

/content/FireRed-OCR/conv_for_infer.py:14: SyntaxWarning: invalid escape sequence '\['
  - Enclose block formulas with,\[,\]. For example:,[,frac{-b,pm,sqrt{b^2 - 4ac}}{2a},]


conv_for_infer imported successfully.
Running on device: cuda
Loading model: FireRedTeam/FireRed-OCR
This may take several minutes on the first run while model weights are downloaded.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

FireRed-OCR model loaded successfully.

Cell 2 complete. Proceed to Cell 3.


## Cell 3 - Storage Setup

Initialises the local invoices.json file that stores all extraction history.
Defines load, save, and delete functions for working with the storage file.

In [4]:
STORAGE_FILE = Path("invoices.json")


def load_history():
    """Read all invoice records from the JSON file. Returns an empty list if the file does not exist."""
    if STORAGE_FILE.exists():
        with open(STORAGE_FILE, "r") as f:
            return json.load(f)
    return []


def save_to_history(record):
    """
    Append a new invoice record to invoices.json.
    Deduplication is based on matching both file_name and invoice_number.
    Returns 'saved' if the record was stored, or 'duplicate' if it already exists.
    """
    history         = load_history()
    new_file_name   = record.get("file_name", "")
    new_invoice_num = record.get("invoice_data", {}).get("invoice_number", "")

    for existing in history:
        existing_file = existing.get("file_name", "")
        existing_num  = existing.get("invoice_data", {}).get("invoice_number", "")
        if existing_file == new_file_name and existing_num == new_invoice_num and new_invoice_num:
            return "duplicate"

    record["id"]        = str(uuid.uuid4())
    record["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    history.append(record)

    with open(STORAGE_FILE, "w") as f:
        json.dump(history, f, indent=2)

    return "saved"


def delete_record(record_id):
    """Delete a single record from history by its UUID. Returns True if found and deleted."""
    history     = load_history()
    new_history = [r for r in history if r.get("id") != record_id]
    if len(new_history) == len(history):
        return False
    with open(STORAGE_FILE, "w") as f:
        json.dump(new_history, f, indent=2)
    return True


# Create the storage file if it does not already exist
if not STORAGE_FILE.exists():
    with open(STORAGE_FILE, "w") as f:
        json.dump([], f)

print(f"Storage file ready at: {STORAGE_FILE.resolve()}")
print(f"Records currently in storage: {len(load_history())}")
print("\nCell 3 complete. Proceed to Cell 4.")

Storage file ready at: /content/invoices.json
Records currently in storage: 0

Cell 3 complete. Proceed to Cell 4.


## Cell 4 - OCR and Parsing Functions

Defines the full invoice processing pipeline:
- Load a file and convert it to a PIL image
- Run FireRed-OCR inference to get structured Markdown
- Parse the Markdown output into a structured invoice dictionary
- Export functions for JSON and CSV download

In [5]:
ALLOWED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".pdf"}


def pdf_to_image(file_path):
    """Convert the first page of a PDF file to a PIL Image using pdf2image at 200 DPI."""
    from pdf2image import convert_from_path
    pages = convert_from_path(file_path, dpi=200, first_page=1, last_page=1)
    if not pages:
        raise ValueError("Could not convert the PDF to an image. Check that poppler-utils is installed.")
    return pages[0]


def load_file_as_image(file_path):
    """Load a JPG, PNG, or PDF file and return it as an RGB PIL Image."""
    ext = Path(file_path).suffix.lower()
    if ext not in ALLOWED_EXTENSIONS:
        raise ValueError(f"Unsupported file type '{ext}'. Please upload a JPG, PNG, or PDF file.")
    if ext == ".pdf":
        return pdf_to_image(file_path)
    return Image.open(file_path).convert("RGB")


def run_ocr(image):
    """
    Run FireRed-OCR inference on a PIL Image.
    Returns the raw Markdown string produced by the model.
    """
    if generate_conv is None:
        raise RuntimeError("conv_for_infer is not available. Check that FireRed-OCR was cloned correctly in Cell 1.")

    # Save the image to a temporary file because generate_conv expects a file path
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
        image.save(tmp.name)
        tmp_path = tmp.name

    try:
        conversation = generate_conv(tmp_path)

        text = processor.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=True,
        )

        from qwen_vl_utils import process_vision_info
        image_inputs, video_inputs = process_vision_info(conversation)

        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to(DEVICE)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=2048,
            )

        # Remove the input prompt tokens from the generated output before decoding
        generated_ids_trimmed = [
            out_ids[len(in_ids):]
            for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]

        output_text = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
        return output_text[0]

    finally:
        os.unlink(tmp_path)


def extract_field(text, *patterns):
    """Try a list of regex patterns against text and return the first match, or None."""
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE | re.MULTILINE)
        if match:
            return match.group(1).strip()
    return None


def parse_invoice(markdown_text):
    """
    Parse the Markdown output from FireRed-OCR into a structured invoice dictionary.
    Fields that cannot be found are set to None rather than an empty string.
    """
    text = markdown_text

    invoice_data = {
        "invoice_number": extract_field(
            text,
            r"invoice\s*(?:no|number|#)[:\s#]*([\w\-/]+)",
            r"inv[\s#\-]*([\w\-/]+)",
        ),
        "date": extract_field(
            text,
            r"(?:invoice\s*)?date[:\s]+([\d]{1,2}[/\-\.][\d]{1,2}[/\-\.][\d]{2,4})",
            r"(?:date)[:\s]+(\w+ \d{1,2},?\s*\d{4})",
            r"(\d{2}[/\-]\d{2}[/\-]\d{4})",
        ),
        "vendor": extract_field(
            text,
            r"(?:from|seller|vendor|billed? by|company)[:\s]+([^\n]+)",
            r"^([A-Z][\w\s&.,'-]+(?:Ltd|Pvt|Inc|Corp|LLC|LLP|Co\.))[^\n]*$",
        ),
        "customer": extract_field(
            text,
            r"(?:to|buyer|customer|billed? to|ship to)[:\s]+([^\n]+)",
            r"(?:client|recipient)[:\s]+([^\n]+)",
        ),
        "gst": extract_field(
            text,
            r"(?:gstin|gst\s*(?:no|number|id))[:\s]+([\w]+)",
            r"(?:tax\s*(?:id|no|number))[:\s]+([\w\-]+)",
        ),
        "total_amount": extract_field(
            text,
            r"(?:grand\s*total|total\s*amount|amount\s*due|total)[:\s]+[\u20b9$\xa3\u20ac]?\s?([\d,]+\.?\d*)",
            r"total[:\s]+([\d,]+\.?\d*)",
        ),
        "tax_amount": extract_field(
            text,
            r"(?:tax|gst|vat|cgst|sgst|igst)[:\s]+[\u20b9$\xa3\u20ac]?\s?([\d,]+\.?\d*)",
            r"(?:tax\s*amount)[:\s]+([\d,]+\.?\d*)",
        ),
        "payment_method": extract_field(
            text,
            r"(?:payment\s*(?:method|mode|terms|via))[:\s]+([^\n]+)",
            r"(?:paid\s*via|pay\s*by)[:\s]+([^\n]+)",
        ),
        "address": extract_field(
            text,
            r"(?:address|billing\s*address)[:\s]+([^\n]+(?:\n[^\n]+){0,2})",
            r"(?:location|place)[:\s]+([^\n]+)",
        ),
        "items": [],
    }

    # Parse line items from Markdown table rows in the format: | desc | qty | unit | total |
    table_rows = re.findall(
        r"\|([^|]+)\|([^|]+)\|([^|]+)\|([^|]+)\|",
        text,
    )
    for row in table_rows:
        desc, qty, unit_price, total = [cell.strip() for cell in row]
        # Skip header rows that contain column labels
        if any(header in desc.lower() for header in ["item", "description", "particular", "product", "---"]):
            continue
        try:
            invoice_data["items"].append({
                "description": desc,
                "qty":         float(re.sub(r"[^\d.]", "", qty)         or "0"),
                "unit_price":  float(re.sub(r"[^\d.]", "", unit_price)  or "0"),
                "total":       float(re.sub(r"[^\d.]", "", total)       or "0"),
            })
        except ValueError:
            continue

    return invoice_data


def process_invoice(file_path):
    """
    Run the complete invoice processing pipeline on an uploaded file.
    Returns a tuple of (invoice_data dict, PIL image, status string, raw markdown text).
    """
    file_name    = Path(file_path).name
    image        = load_file_as_image(file_path)
    raw_markdown = run_ocr(image)
    invoice_data = parse_invoice(raw_markdown)

    record = {
        "file_name":    file_name,
        "invoice_data": invoice_data,
    }
    status = save_to_history(record)

    return invoice_data, image, status, raw_markdown


def export_json():
    """Return the path to the invoices.json file so Gradio can offer it as a download."""
    return str(STORAGE_FILE)


def export_csv():
    """Flatten the invoice history into a CSV file and return its path."""
    history = load_history()
    rows = []
    for r in history:
        d = r.get("invoice_data", {})
        rows.append({
            "id":             r.get("id"),
            "timestamp":      r.get("timestamp"),
            "file_name":      r.get("file_name"),
            "invoice_number": d.get("invoice_number"),
            "date":           d.get("date"),
            "vendor":         d.get("vendor"),
            "customer":       d.get("customer"),
            "gst":            d.get("gst"),
            "total_amount":   d.get("total_amount"),
            "tax_amount":     d.get("tax_amount"),
            "payment_method": d.get("payment_method"),
            "address":        d.get("address"),
        })
    df = pd.DataFrame(rows)
    csv_path = "invoices_export.csv"
    df.to_csv(csv_path, index=False)
    return csv_path


print("OCR and parsing functions defined.")
print("\nCell 4 complete. Proceed to Cell 5.")

OCR and parsing functions defined.

Cell 4 complete. Proceed to Cell 5.


## Cell 5 - UI Data Formatting Functions

Converts raw invoice data into DataFrames and Plotly chart figures
that Gradio can render directly inside the interface.

In [6]:
def invoice_data_to_df(invoice_data):
    """Convert the invoice_data dictionary into a two-column Field/Value DataFrame."""
    scalar_fields = [
        "invoice_number",
        "date",
        "vendor",
        "customer",
        "gst",
        "total_amount",
        "tax_amount",
        "payment_method",
        "address",
    ]
    rows = []
    for field in scalar_fields:
        label = field.replace("_", " ").title()
        value = invoice_data.get(field)
        rows.append({"Field": label, "Value": value if value is not None else "-"})
    return pd.DataFrame(rows)


def items_to_df(invoice_data):
    """Convert the line items list from invoice_data into a DataFrame with four columns."""
    items = invoice_data.get("items", [])
    if not items:
        return pd.DataFrame(columns=["Description", "Qty", "Unit Price", "Total"])
    return pd.DataFrame([
        {
            "Description": item.get("description", "-"),
            "Qty":         item.get("qty", 0),
            "Unit Price":  item.get("unit_price", 0),
            "Total":       item.get("total", 0),
        }
        for item in items
    ])


def history_to_df(search_query="", date_filter=""):
    """Load all history and return a filtered summary DataFrame for the History tab."""
    history = load_history()
    rows = []

    for r in history:
        d      = r.get("invoice_data", {})
        vendor = d.get("vendor") or "-"
        date   = d.get("date")   or "-"

        if search_query and search_query.lower() not in vendor.lower():
            continue
        if date_filter and date_filter not in date:
            continue

        rows.append({
            "Timestamp":    r.get("timestamp", "-"),
            "File":         r.get("file_name", "-"),
            "Invoice No":   d.get("invoice_number") or "-",
            "Vendor":       vendor,
            "Customer":     d.get("customer") or "-",
            "Total Amount": d.get("total_amount") or "-",
            "Date":         date,
        })

    if not rows:
        return pd.DataFrame(columns=["Timestamp", "File", "Invoice No", "Vendor", "Customer", "Total Amount", "Date"])
    return pd.DataFrame(rows)


def get_analytics():
    """Compute totals and build a Plotly bar chart of invoices processed per day."""
    history        = load_history()
    total_invoices = len(history)
    total_value    = 0.0

    for r in history:
        raw     = r.get("invoice_data", {}).get("total_amount") or "0"
        cleaned = re.sub(r"[^\d.]", "", str(raw))
        try:
            total_value += float(cleaned)
        except ValueError:
            pass

    # Build the recent uploads table from the last 10 records
    recent = history[-10:][::-1]
    recent_rows = [
        {
            "File":         r.get("file_name", "-"),
            "Vendor":       r.get("invoice_data", {}).get("vendor") or "-",
            "Total Amount": r.get("invoice_data", {}).get("total_amount") or "-",
            "Timestamp":    r.get("timestamp", "-"),
        }
        for r in recent
    ]
    recent_df = pd.DataFrame(recent_rows) if recent_rows else pd.DataFrame()

    # Build bar chart showing invoice counts grouped by day
    dates = [r.get("timestamp", "")[:10] for r in history if r.get("timestamp")]

    if dates:
        date_counts = pd.Series(dates).value_counts().sort_index().reset_index()
        date_counts.columns = ["Date", "Count"]
        fig = px.bar(
            date_counts,
            x="Date",
            y="Count",
            title="Invoices Processed Per Day",
            color_discrete_sequence=["#6366f1"],
        )
        fig.update_layout(
            plot_bgcolor="rgba(0,0,0,0)",
            paper_bgcolor="rgba(0,0,0,0)",
        )
    else:
        fig = px.bar(title="No data yet. Extract some invoices first.")

    return total_invoices, round(total_value, 2), recent_df, fig


print("UI formatting functions defined.")
print("\nCell 5 complete. Proceed to Cell 6.")

UI formatting functions defined.

Cell 5 complete. Proceed to Cell 6.


## Cell 6 - Build Gradio Interface

Constructs the complete Gradio application with four tabs:
- Extract Invoice
- History
- Analytics
- About

Running this cell does not launch the app yet. The launch happens in Cell 7.

In [7]:
# --- Event handler functions -------------------------------------------------

def handle_extraction(file):
    """
    Triggered when the user clicks the Extract button.
    Returns outputs to update image preview, fields table, items table,
    JSON display, status message, and raw OCR text box.
    """
    empty_fields = pd.DataFrame(columns=["Field", "Value"])
    empty_items  = pd.DataFrame(columns=["Description", "Qty", "Unit Price", "Total"])

    if file is None:
        return None, empty_fields, empty_items, "{}", "Please upload a file first.", ""

    try:
        invoice_data, image, status, raw_md = process_invoice(file.name)

        fields_df = invoice_data_to_df(invoice_data)
        items_df  = items_to_df(invoice_data)
        json_str  = json.dumps(invoice_data, indent=2)

        if status == "duplicate":
            msg = "Duplicate invoice detected. This invoice is already in history. Showing extracted data."
        else:
            msg = "Invoice extracted and saved to history successfully."

        return image, fields_df, items_df, json_str, msg, raw_md

    except ValueError as e:
        return None, empty_fields, empty_items, "{}", f"Validation error: {e}", ""
    except Exception as e:
        return None, empty_fields, empty_items, "{}", f"Unexpected error: {e}", ""


def handle_search(query, date_filter):
    """Filter the history table based on vendor search and date filter inputs."""
    return history_to_df(search_query=query, date_filter=date_filter)


def handle_analytics_refresh():
    """Recompute and return all analytics data for display in the Analytics tab."""
    total, value, recent_df, fig = get_analytics()
    total_text = f"**Total Invoices Processed**\n\n# {total}"
    value_text = f"**Total Value Extracted**\n\n# {value:,.2f}"
    return total_text, value_text, recent_df, fig


# --- Custom CSS applied to the Gradio container ------------------------------

custom_css = """
.gradio-container {
    font-family: Inter, sans-serif;
}
footer {
    display: none !important;
}
"""

# --- Build the Gradio interface ----------------------------------------------

with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="violet", neutral_hue="slate"),
    css=custom_css,
    title="Invoice Extractor",
) as demo:

    gr.Markdown("""
    # AI Invoice Extraction System
    Powered by **FireRedTeam/FireRed-OCR** - Upload an invoice and extract structured data instantly.
    Supports JPG, PNG, and PDF files.
    """)

    with gr.Tabs():

        # ------------------------------------------------------------------
        # Tab 1 - Extract Invoice
        # ------------------------------------------------------------------
        with gr.TabItem("Extract Invoice"):

            with gr.Row():

                # Left column - file upload and invoice image preview
                with gr.Column(scale=1):
                    gr.Markdown("### Upload")
                    file_input = gr.File(
                        label="Upload invoice (JPG, PNG, or PDF)",
                        file_types=[".jpg", ".jpeg", ".png", ".pdf"],
                    )
                    extract_btn   = gr.Button("Extract Invoice", variant="primary", size="lg")
                    status_output = gr.Textbox(label="Status", interactive=False)
                    image_preview = gr.Image(label="Invoice Preview", height=420)

                # Right column - extracted fields and line items
                with gr.Column(scale=1):
                    gr.Markdown("### Extracted Fields")
                    gr.Markdown("Click any value in the table below to edit it before downloading.")
                    fields_table = gr.Dataframe(
                        headers=["Field", "Value"],
                        interactive=True,
                        wrap=True,
                        label="Invoice Fields",
                    )
                    gr.Markdown("### Line Items")
                    items_table = gr.Dataframe(
                        headers=["Description", "Qty", "Unit Price", "Total"],
                        interactive=True,
                        label="Line Items",
                    )

            with gr.Row():
                with gr.Column():
                    gr.Markdown("### Structured JSON Output")
                    json_output = gr.Code(
                        language="json",
                        label="Extracted JSON",
                    )

                with gr.Column():
                    gr.Markdown("### Export")
                    download_json_btn  = gr.Button("Download All History as JSON")
                    download_csv_btn   = gr.Button("Download All History as CSV")
                    json_download_file = gr.File(label="JSON file", visible=False)
                    csv_download_file  = gr.File(label="CSV file",  visible=False)

            with gr.Accordion("Raw OCR Output (for debugging)", open=False):
                raw_output = gr.Textbox(
                    label="Raw Markdown from FireRed-OCR",
                    lines=15,
                    interactive=False,
                )

            # Wire up the Extract button
            extract_btn.click(
                fn=handle_extraction,
                inputs=[file_input],
                outputs=[image_preview, fields_table, items_table, json_output, status_output, raw_output],
            )

            # Wire up the download buttons
            download_json_btn.click(
                fn=lambda: gr.update(value=export_json(), visible=True),
                outputs=[json_download_file],
            )
            download_csv_btn.click(
                fn=lambda: gr.update(value=export_csv(), visible=True),
                outputs=[csv_download_file],
            )

        # ------------------------------------------------------------------
        # Tab 2 - History
        # ------------------------------------------------------------------
        with gr.TabItem("History"):
            gr.Markdown("### All Extracted Invoices")

            with gr.Row():
                search_box   = gr.Textbox(placeholder="Search by vendor name ...",    label="Vendor Search", scale=3)
                date_box     = gr.Textbox(placeholder="Filter by date e.g. 2026-05", label="Date Filter",   scale=2)
                refresh_hist = gr.Button("Refresh", scale=1)

            history_table = gr.Dataframe(
                value=history_to_df(),
                interactive=False,
                wrap=True,
                label="Invoice History",
            )

            with gr.Row():
                hist_json_btn  = gr.Button("Export History JSON")
                hist_csv_btn   = gr.Button("Export History CSV")
                hist_json_file = gr.File(label="JSON", visible=False)
                hist_csv_file  = gr.File(label="CSV",  visible=False)

            search_box.change(fn=handle_search,  inputs=[search_box, date_box], outputs=[history_table])
            date_box.change(fn=handle_search,    inputs=[search_box, date_box], outputs=[history_table])
            refresh_hist.click(fn=handle_search, inputs=[search_box, date_box], outputs=[history_table])

            hist_json_btn.click(
                fn=lambda: gr.update(value=export_json(), visible=True),
                outputs=[hist_json_file],
            )
            hist_csv_btn.click(
                fn=lambda: gr.update(value=export_csv(), visible=True),
                outputs=[hist_csv_file],
            )

        # ------------------------------------------------------------------
        # Tab 3 - Analytics
        # ------------------------------------------------------------------
        with gr.TabItem("Analytics"):
            gr.Markdown("### Dashboard")
            refresh_analytics = gr.Button("Refresh Analytics", variant="secondary")

            with gr.Row():
                stat_invoices = gr.Markdown("**Total Invoices Processed**\n\n# 0")
                stat_value    = gr.Markdown("**Total Value Extracted**\n\n# 0.00")

            gr.Markdown("### Recent Uploads")
            recent_table = gr.Dataframe(interactive=False, label="Last 10 Invoices")

            gr.Markdown("### Invoices Per Day")
            chart = gr.Plot(label="Volume Chart")

            refresh_analytics.click(
                fn=handle_analytics_refresh,
                outputs=[stat_invoices, stat_value, recent_table, chart],
            )
            # Auto-load analytics data when the page opens
            demo.load(
                fn=handle_analytics_refresh,
                outputs=[stat_invoices, stat_value, recent_table, chart],
            )

        # ------------------------------------------------------------------
        # Tab 4 - About
        # ------------------------------------------------------------------
        with gr.TabItem("About"):
            gr.Markdown("""
            ## About This System

            | Component       | Detail                               |
            |-----------------|--------------------------------------|
            | OCR Model       | FireRedTeam/FireRed-OCR              |
            | Architecture    | Qwen3-VL Vision-Language Model       |
            | UI Framework    | Gradio                               |
            | Storage         | Local invoices.json                  |
            | Supported Files | JPG, PNG, PDF                        |

            ### How It Works

            1. Upload an invoice image or PDF using the Extract tab.
            2. PDF files are converted to images using pdf2image at 200 DPI.
            3. The image is passed to FireRed-OCR which returns structured Markdown.
            4. The Markdown is parsed with regex patterns to extract invoice fields.
            5. The result is saved to invoices.json with duplicate detection.
            6. Extracted fields appear in an editable table.

            ### Tips

            - Use clear, high-resolution scans for the best extraction accuracy.
            - PDF extraction only processes the first page.
            - All extracted fields are editable in the table before you download.
            - Use the vendor search box in the History tab to filter records.
            - The invoices.json file in the same directory stores all your history.
            - On Colab, history is lost when the session ends unless you download it.
            """)


print("Gradio app built successfully.")
print("\nCell 6 complete. Proceed to Cell 7 to launch.")

/tmp/ipykernel_2928/1118863787.py:61: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_2928/1118863787.py:61: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


Gradio app built successfully.

Cell 6 complete. Proceed to Cell 7 to launch.


## Cell 7 - Launch the App

Starts the Gradio server.

**On Google Colab**: the app uses `share=True` which creates a public tunnel link valid for 72 hours.
Look for the line that says `Running on public URL: https://....gradio.live` and open that link.

**On a local machine**: opens on http://localhost:7860 in your browser.

In [8]:
import os

# Detect whether the notebook is running inside Google Colab
running_on_colab = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if running_on_colab:
    print("Google Colab detected.")
    print("Launching with share=True ...")
    print("Wait for the line: Running on public URL: https://....gradio.live")
    print("Open that link in your browser to use the app.")
    print()
    demo.launch(
        share=True,
        show_error=True,
        quiet=False,
    )
else:
    print("Local environment detected.")
    print("Launching on http://localhost:7860")
    print()
    demo.launch(
        share=False,
        server_name="0.0.0.0",
        server_port=7860,
        show_error=True,
    )

Google Colab detected.
Launching with share=True ...
Wait for the line: Running on public URL: https://....gradio.live
Open that link in your browser to use the app.

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://91d592c2df1fd18176.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
